![image.png](https://i.imgur.com/a3uAqnb.png)

# Text Generation / Mini Chatbot with Transformers

## 🎯 Objective
Fine-tune a pre-trained **Causal Language Model (GPT-2 / DistilGPT-2)** on a dialogue dataset to build a **mini chatbot** that can hold simple conversations.

## 📚 What You'll Learn
- **Causal Language Modeling (CLM)** vs Masked Language Modeling (MLM)
- How Decoder-only models (GPT family) work
- Fine-tuning on conversational data
- Different **text generation strategies**: greedy, beam search, top-k, top-p (nucleus), temperature
- Building an interactive chatbot loop
- Creating a **web UI with Gradio** for the chatbot

---

## Step 1: Install required libraries

In [ ]:
!pip install transformers datasets accelerate -q

## Step 2: Import libraries

In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline
)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


## Step 3: Understanding Causal Language Modeling (CLM)

Causal Language Modeling means the model predicts the **next token** based on the previous tokens only (no future context, unlike BERT).

Examples of CLM models: **GPT-2, GPT-3, LLaMA, Mistral**

Example:
```
Input:  "The weather today is"
Output: "The weather today is very nice and sunny."
```

The model is trained to predict each token from the previous tokens only.

## Step 4: Load the DailyDialog dataset

**DailyDialog** contains thousands of natural daily conversations - perfect for training a chatbot.

In [ ]:
dataset = load_dataset("DeepPavlov/daily_dialog")
print(dataset)

# View a sample conversation
print("\n=== Sample conversation ===")
for utterance in dataset["train"][0]["dialog"]:
    print(f"- {utterance}")

## Step 5: Format the dialogues into a chatbot-friendly format

We'll use special tokens `<|user|>` and `<|bot|>` to distinguish between participants, and `<|endoftext|>` to end each dialogue.

In [ ]:
def format_dialog(example):
    """Convert each dialog into a single text string."""
    dialog = example["dialog"]
    formatted = ""
    for i, utterance in enumerate(dialog):
        speaker = "<|user|>" if i % 2 == 0 else "<|bot|>"
        formatted += f"{speaker} {utterance.strip()}\n"
    formatted += "<|endoftext|>"
    return {"text": formatted}

# Use a subset for fast training
...
...

# Show an example after formatting
print("=== Example after formatting ===")
print(train_data[0]["text"])

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

=== Example after formatting ===
<|user|> Say , Jim , how about going for a few beers after dinner ?
<|endoftext|>


## Step 6: Load the tokenizer and add special tokens

In [ ]:
model_checkpoint = "distilgpt2"  # Smaller, faster version of GPT-2
tokenizer = ...


print(f"Vocab size before adding new tokens: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

# Add special tokens
special_tokens = {
    "additional_special_tokens": ["<|user|>", "<|bot|>"],
    "pad_token": "<|pad|>"
}
....


print(f"Vocab size after adding new tokens: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}")
print(f"EOS token: {tokenizer.eos_token}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size after adding new tokens: 50260
Pad token: <|pad|>
EOS token: <|endoftext|>


## Step 7: Tokenize the data

In [ ]:
max_length = ...

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )

train_tokenized = ...

eval_tokenized = ...

# In CLM, labels = input_ids (the model predicts each next token)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=...,
    mlm=...
)

print("Tokenization done ✅")

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenization done ✅


## Step 8: Load the model and resize the embedding layer

Since we added new tokens, we must resize the embedding layer to fit the new vocabulary size.

In [ ]:
model = ...

print(f"Model parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 81,914,880


## Step 9: Train the model

In [ ]:
training_args = ...

trainer = ...

trainer.train()

## Step 10: Evaluate the model with Perplexity

**Perplexity** is the standard metric for language models. The lower, the better.
- Perplexity = exp(loss)
- A perplexity of 10 means the model is choosing on average from 10 equally likely options for each next token.

In [ ]:
import math

eval_results =...

## Step 11: Understanding Text Generation Strategies

There are multiple strategies for generating text:

| Strategy | Description | When to use? |
|----------|-------------|--------------|
| **Greedy** | Always picks the highest-probability token | Deterministic but boring |
| **Beam Search** | Explores multiple paths and picks the best one | More coherent text |
| **Top-k** | Samples from the top k most likely tokens | More variety |
| **Top-p (Nucleus)** | Samples from smallest token set whose probability sums to p | The most natural |
| **Temperature** | Controls randomness (low = conservative, high = creative) | Tune freedom |

In [ ]:
model = trainer.model
model.eval()


def generate_text(prompt, strategy="greedy", max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Specify the generation strategy for each strategy type.
    # Greedy: always pick the highest-probability token
    if strategy == "greedy":

        outputs = ...
    # Beam Search:	Explores multiple paths and picks the best one
    elif strategy == "beam":

        outputs = ...
    # Top-k:	Samples from the top k most likely tokens
    elif strategy == "top_k":

        outputs = ...
    # Top-p:	Samples from smallest token set whose probability sums to p
    elif strategy == "top_p":

        outputs = ...

    return tokenizer.decode(outputs[0], skip_special_tokens=False)

# Compare the different strategies
prompt = "<|user|> Hi, how are you today?\n<|bot|>"

print("GREEDY:")
print(generate_text(prompt, "greedy"))
print("\nBEAM SEARCH:")
print(generate_text(prompt, "beam"))
print("\nTOP-K SAMPLING:")
print(generate_text(prompt, "top_k"))
print("\nTOP-P (NUCLEUS):")
print(generate_text(prompt, "top_p"))

## Step 12: Effect of Temperature on output

Temperature controls the level of "creativity" in the model:
- **Low temperature (0.1-0.3)**: very predictable, conservative output
- **Medium temperature (0.7-1.0)**: balanced
- **High temperature (1.5+)**: very creative but maybe incoherent

In [ ]:
prompt = "<|user|> Tell me about your weekend plans.\n<|bot|>"

# Generate a reply for each temperature value and observe how the output changes.
for temp in [0.3, 0.7, 1.0, 1.5]:
    ...
    ...
    ...


    text = tokenizer.decode(..., skip_special_tokens=False)
    print(f"\n Temperature = {temp}")
    print(text)

## Step 13: Test the chatbot with a sample conversation

In [ ]:

test_messages = [
    "Hello, how are you?",
    "What did you do today?",
    "That sounds fun! Do you like sports?",
    "What's your favorite food?"
]

...
...
...


## Step 14: Build the Mini Chatbot 🤖

Create an interactive chatbot using Gradio [ Documentation](https://www.gradio.app/)

## 📝 For further Exploration

1. **Compare base model vs fine-tuned model**: load the original DistilGPT-2 and compare its replies to the fine-tuned version.
2. **Train on a different dataset**: try `blended_skill_talk` or `empathetic_dialogues` and observe the differences in the chatbot's personality.
3. **Try a stronger model**: use `gpt2-medium` or `microsoft/DialoGPT-small` and compare the quality.
4. **Tune generation parameters**: you can use Gradio sliders to find the best combo of `temperature` and `top_p` for natural conversations.
5. **Build a topic-specific chatbot**: prepare a small custom dataset (e.g., a customer support bot or a coding tutor) and fine-tune the model on it.
6. **Add a system prompt**: at the start of each conversation, add a personality prompt like `<|system|> You are a helpful assistant who loves science.`


## Concepts to Discuss
- What's the difference between **Causal LM** (GPT) and **Masked LM** (BERT)?
- Why do we use `mlm=False` in the data collator?
- What does **Perplexity** measure and why is it a suitable metric?
- What's the trade-off between **Greedy** and **Sampling**?
- Why do we need to resize the embedding layer after adding new tokens?
- What is the role of `no_repeat_ngram_size` in preventing repetition?
